# 04 — Train and independently test corrected verifier LoRAs

Run one model per Biowulf job using `JAMIA_MODEL_KEY`. The legacy adapter supplies only its documented LoRA architecture and provenance; its weights are **not** used to initialize the corrected adapter. Each corrected adapter is trained on the grouped train split, selected on validation loss, and scored once on the untouched grouped test split.

This notebook is restart-safe. Cell 2 skips fine-tuning when it finds a complete corrected adapter whose weight file and training provenance match `model_key` and the canonical training base. Partial output is moved to a recoverable timestamped quarantine directory before a clean run; contradictory completed provenance remains a hard stop. The independent test appends and validates one prediction at a time, resumes only missing verifier records, and skips model loading when both evaluation artifacts are already complete. CPU offload is prohibited during training but permitted and recorded during evaluation; disk offload remains prohibited.

The four text-model profiles in this revision reproduce their supplied reference notebooks: GPT-OSS-20B, Llama-3-8B, Med-Qwen2-7B, and OpenBioLLM-8B use `AutoModelForCausalLM` with `torch_dtype=bfloat16` and no bitsandbytes configuration. GPT-OSS retains its checkpoint-native MXFP4 weights. The existing `llama_3_1_8b` job key is retained only for script compatibility; its audited checkpoint is `unsloth/llama-3-8b-Instruct` and all outputs identify it as Llama-3-8B, not Llama 3.1.

In [ ]:
from pathlib import Path
import json, os, sys

def find_rerun_dir():
    candidates = [
        Path(os.environ.get("JAMIA_RERUN_DIR", "")),
        Path.cwd(),
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if str(candidate) and (candidate / "rerun_config.json").exists():
            return candidate.resolve()
    raise FileNotFoundError("Set JAMIA_RERUN_DIR to the folder containing rerun_config.json")

RERUN_DIR = find_rerun_dir()
# Never expose the implementation directory as a top-level import location:
# rerun_code/statistics.py would shadow Python's standard-library statistics.
implementation_dir = (RERUN_DIR / "src" / "rerun_code").resolve()
clean_sys_path = []
for entry in sys.path:
    try:
        resolved_entry = Path(entry or ".").resolve()
    except Exception:
        resolved_entry = None
    if resolved_entry != implementation_dir:
        clean_sys_path.append(entry)
sys.path[:] = clean_sys_path
sys.path.insert(0, str(RERUN_DIR / "src"))
from rerun_code.config import load_config, output_paths
RERUN_DIR, CONFIG = load_config(RERUN_DIR)
PATHS = output_paths(CONFIG)
print("Code:", RERUN_DIR)
print("Output:", PATHS["root"])

In [ ]:
import pandas as pd
from rerun_code.common import read_jsonl, write_json
from rerun_code.config import sha256_path
from rerun_code.modeling import adapter_config, completed_corrected_adapter, IncompleteCorrectedAdapterError, quarantine_incomplete_corrected_adapter, release_accelerator_memory, resolve_training_base_model, train_corrected_verifier, load_processor_and_model, assert_model_identity
from rerun_code.generation import ModelRunner
from rerun_code.verifier_data import verifier_prompt

import importlib.metadata as package_metadata
print("Python:", sys.version.split()[0])
runtime_packages = {}
for package in ("torch", "transformers", "kernels", "peft", "accelerate", "bitsandbytes", "safetensors", "huggingface-hub"):
    try: runtime_packages[package] = package_metadata.version(package)
    except package_metadata.PackageNotFoundError: runtime_packages[package] = "NOT INSTALLED"
    print(package, runtime_packages[package])

model_key = os.environ.get("JAMIA_MODEL_KEY", "qwen2_1_5b")
if model_key not in CONFIG["models"]: raise KeyError(model_key)
spec = CONFIG["models"][model_key]
legacy_config = adapter_config(spec["legacy_adapter"])
training_base = resolve_training_base_model(spec)
print("Legacy adapter base (audit only):", legacy_config.get("base_model_name_or_path"))
print("Fresh corrected-training base:", training_base)
print("Loader profile:", spec.get("loader_profile", "default"))
print("Tokenizer source:", spec.get("tokenizer_source", "base"))
print("Chat-template policy:", spec.get("chat_template_policy", "auto"))
print("Reference loading notebook:", spec.get("reference_loading_notebook", "not registered"))
if spec.get("manuscript_identity_note"):
    print("MANUSCRIPT IDENTITY NOTE:", spec["manuscript_identity_note"])
train = pd.DataFrame(read_jsonl(PATHS["verifier_data"] / "train.jsonl"))
validation = pd.DataFrame(read_jsonl(PATHS["verifier_data"] / "validation.jsonl"))
test = pd.DataFrame(read_jsonl(PATHS["verifier_data"] / "test.jsonl"))
model_out = PATHS["verifiers"] / model_key
try:
    completed = completed_corrected_adapter(model_out, model_key, training_base, spec=spec)
except IncompleteCorrectedAdapterError as exc:
    quarantined = quarantine_incomplete_corrected_adapter(exc.output_dir)
    print("INCOMPLETE PRIOR TRAINING OUTPUT QUARANTINED:", quarantined)
    print("The directory was moved, not deleted. Starting a clean run for:", training_base)
    completed = None
if completed is not None:
    adapter_dir = completed["adapter_dir"]
    actual_base = completed["actual_base"]
    history = completed["history"]
    print("FINE-TUNING SKIPPED: audited corrected adapter already exists.")
    print("Corrected adapter:", adapter_dir)
    print("Corrected adapter weights:", completed["weights_file"])
    if completed["adapter_base_before_normalization"] != actual_base:
        print(
            "Normalized saved adapter base from",
            completed["adapter_base_before_normalization"], "to", actual_base,
        )
else:
    print("No completed corrected adapter found; starting fine-tuning.")
    adapter_dir, actual_base, history = train_corrected_verifier(model_key, spec, train, validation, model_out, verifier_prompt)
    write_json(model_out / "training_provenance.json", {
        "model_key": model_key, "actual_training_base": actual_base,
        "legacy_adapter_base_audit_only": legacy_config.get("base_model_name_or_path"),
        "legacy_adapter_used_for_configuration_only": spec["legacy_adapter"],
        "legacy_adapter_config_sha256": sha256_path(Path(spec["legacy_adapter"]) / "adapter_config.json"),
        "legacy_weights_loaded": False,
        "loader_profile": spec.get("loader_profile", "default"),
        "tokenizer_source": spec.get("tokenizer_source", "base"),
        "chat_template_policy": spec.get("chat_template_policy", "auto"),
        "reference_loading_notebook": spec.get("reference_loading_notebook"),
        "display_name": spec["display_name"],
        "manuscript_identity_note": spec.get("manuscript_identity_note"),
        "runtime_packages": runtime_packages,
        "history": history
    })
    print("Saved corrected adapter:", adapter_dir)

independent_predictions_path = model_out / "independent_test_predictions.jsonl"
independent_metrics_path = model_out / "independent_test_metrics.json"
missing_evaluation_artifacts = [
    str(path) for path in (independent_predictions_path, independent_metrics_path)
    if not path.exists()
]
if missing_evaluation_artifacts:
    print("INDEPENDENT EVALUATION REQUIRED. Missing:", missing_evaluation_artifacts)
else:
    print("Independent-evaluation artifacts exist and will be audited in the next cell.")

In [ ]:
def audit_saved_independent_predictions(path, test_frame):
    expected = {
        str(row["verifier_record_id"]): {
            "truth": bool(row["verdict"]),
            "patient_or_source_group": str(row["patient_or_source_group"]),
        }
        for _, row in test_frame.iterrows()
    }
    saved = read_jsonl(path) if path.exists() else []
    completed_ids = set()
    for line_number, row in enumerate(saved, start=1):
        record_id = str(row.get("verifier_record_id") or "")
        if record_id not in expected:
            raise RuntimeError(
                f"Unknown verifier_record_id in {path}:{line_number}: {record_id!r}"
            )
        if record_id in completed_ids:
            raise RuntimeError(f"Duplicate verifier_record_id in {path}: {record_id}")
        if bool(row.get("truth")) != expected[record_id]["truth"]:
            raise RuntimeError(f"Truth mismatch for {record_id} in {path}")
        if str(row.get("patient_or_source_group") or "") != expected[record_id]["patient_or_source_group"]:
            raise RuntimeError(f"Group mismatch for {record_id} in {path}")
        if row.get("verdict") not in (True, False):
            raise RuntimeError(
                f"Saved verifier output for {record_id} is unparseable. "
                "Move the independent prediction file aside before retrying."
            )
        completed_ids.add(record_id)
    return completed_ids, saved

def score_independent_predictions(rows, evaluation_device_map, evaluation_cpu_offload):
    scored = pd.DataFrame(rows)
    if len(scored) != len(test):
        raise AssertionError(
            f"Independent evaluation has {len(scored)} predictions; expected {len(test)}"
        )
    if scored["verdict"].isna().any():
        raise AssertionError(f"Unparseable verifier outputs: {scored['verdict'].isna().sum()}")
    truth, pred = scored["truth"].astype(bool), scored["verdict"].astype(bool)
    tp = int((truth & pred).sum())
    tn = int((~truth & ~pred).sum())
    fp = int((~truth & pred).sum())
    fn = int((truth & ~pred).sum())
    return {
        "model_key": model_key,
        "actual_base_model": actual_base,
        "n": len(scored),
        "accuracy": (tp + tn) / len(scored),
        "sensitivity": tp / (tp + fn) if tp + fn else None,
        "specificity": tn / (tn + fp) if tn + fp else None,
        "f1": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else None,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "evaluation_cpu_offload": evaluation_cpu_offload,
        "evaluation_device_map": evaluation_device_map,
        "test_split_sha256": sha256_path(PATHS["verifier_data"] / "test.jsonl"),
    }

completed_ids, saved_predictions = audit_saved_independent_predictions(
    independent_predictions_path, test
)
expected_ids = set(test["verifier_record_id"].astype(str))
pending_ids = expected_ids - completed_ids
metrics_complete = independent_metrics_path.exists()
if not pending_ids and metrics_complete:
    existing_metrics = json.loads(independent_metrics_path.read_text(encoding="utf-8"))
    if int(existing_metrics.get("n") or 0) != len(test):
        raise RuntimeError(
            f"Existing independent metrics report n={existing_metrics.get('n')}; "
            f"expected {len(test)}. Move {independent_metrics_path} aside and rerun."
        )
    print("INDEPENDENT EVALUATION SKIPPED: predictions and metrics are complete.")
    print(json.dumps(existing_metrics, indent=2))
else:
    print(
        f"Independent evaluation resume audit: complete={len(completed_ids)}, "
        f"pending={len(pending_ids)}, metrics_complete={metrics_complete}"
    )
    recorded_adapter_base = adapter_config(adapter_dir).get("base_model_name_or_path")
    print("Corrected adapter-recorded base:", recorded_adapter_base)
    print("Reloading audited canonical base:", actual_base)
    release_accelerator_memory()
    processor, tokenizer, model, actual_base = load_processor_and_model(
        spec,
        adapter_path=adapter_dir,
        base_model_override=actual_base,
    )
    assert_model_identity(model_key, spec, actual_base)
    loaded_backbone = model.get_base_model() if hasattr(model, "get_base_model") else model
    evaluation_device_map = {
        str(key): str(value)
        for key, value in (getattr(loaded_backbone, "hf_device_map", {}) or {}).items()
    }
    evaluation_cpu_offload = any(
        value.lower() == "cpu" for value in evaluation_device_map.values()
    )
    print("Evaluation CPU offload:", evaluation_cpu_offload)
    runner = ModelRunner(
        processor, tokenizer, model, spec["architecture"], CONFIG["generation"],
        actual_base=actual_base,
    )
    progress_every = max(1, int(os.environ.get("JAMIA_VERIFIER_PROGRESS_EVERY", "25")))
    generated = 0
    for _, row in test.iterrows():
        record_id = str(row["verifier_record_id"])
        if record_id in completed_ids:
            continue
        result = runner.verify(
            str(row["report_text"]),
            adapter_enabled=True,
            image_path=str(row.get("image_path", "") or ""),
        )
        if result.get("verdict") not in (True, False):
            raise RuntimeError(
                f"Unparseable verifier output for {record_id}: {result.get('raw')!r}. "
                "This record was not saved."
            )
        output = {
            "verifier_record_id": record_id,
            "patient_or_source_group": str(row["patient_or_source_group"]),
            "truth": bool(row["verdict"]),
            **result,
        }
        with independent_predictions_path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(output, ensure_ascii=False) + "\n")
            handle.flush()
        completed_ids.add(record_id)
        generated += 1
        if generated % progress_every == 0 or not (expected_ids - completed_ids):
            print(
                f"Independent evaluation progress: generated={generated}/{len(pending_ids)}, "
                f"total_complete={len(completed_ids)}/{len(expected_ids)}"
            )
    if completed_ids != expected_ids:
        raise AssertionError(
            f"Independent evaluation ended with {len(expected_ids - completed_ids)} missing records"
        )
    final_predictions = read_jsonl(independent_predictions_path)
    summary = score_independent_predictions(
        final_predictions, evaluation_device_map, evaluation_cpu_offload
    )
    write_json(independent_metrics_path, summary)
    print("Saved independent predictions:", independent_predictions_path)
    print("Saved independent metrics:", independent_metrics_path)
    print(json.dumps(summary, indent=2))

Repeat this notebook for all seven keys: `medgemma_4b`, `phi4_multimodal`, `gpt_oss_20b`, `llama_3_1_8b`, `medqwen2_7b`, `openbiollm_8b`, and `qwen2_1_5b`. The compatibility key `llama_3_1_8b` writes Llama-3-8B identity into its outputs because that is the checkpoint used by the supplied reference notebook and adapter.